# 03 - Labels and Leakage

In [1]:

from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
PROCESSED = ROOT / "data/processed/synthetic_h50"
print("Repository:", ROOT)
print("Processed data:", PROCESSED)


Repository: /home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting
Processed data: /home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting/data/processed/synthetic_h50


In [2]:
from lob_project.data.processed import load_processed
from lob_project.data.split import assert_no_window_overlap
from lob_project.labels.cost_aware import cost_aware_direction_labels

data = load_processed(PROCESSED)
sequence_length = data.metadata["sequence_length"]
assert_no_window_overlap(data.train_indices, data.validation_indices, sequence_length)
assert_no_window_overlap(data.validation_indices, data.test_indices, sequence_length)
print("No input-window overlap across adjacent splits.")

No input-window overlap across adjacent splits.


In [3]:
boundary_table = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "first_target": [data.train_indices.min(), data.validation_indices.min(), data.test_indices.min()],
    "last_target": [data.train_indices.max(), data.validation_indices.max(), data.test_indices.max()],
    "count": [len(data.train_indices), len(data.validation_indices), len(data.test_indices)],
})
boundary_table

,split,first_target,last_target,count
0,train,99,5549,5451
1,validation,5699,6749,1051
2,test,6899,7949,1051


In [4]:
source = np.load(PROCESSED / "source_features.npy", mmap_mode="r")
best_ask, best_bid = source[:, 0], source[:, 2]
cost_labels = cost_aware_direction_labels(
    best_ask, best_bid, horizon=data.metadata["horizon"], fee_bps=0.2
)
comparison = pd.crosstab(
    pd.Series(np.asarray(data.labels), name="benchmark"),
    pd.Series(cost_labels, name="cost-aware"),
    normalize="index",
)
comparison

cost-aware,0,1,2
benchmark,,,
0,0.924891,0.020661,0.054448
1,0.539589,0.228739,0.231672
2,0.091961,0.025106,0.882934
